# 08 Decorator | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: eksplozja klas bez dekoratora
2. 🎁 Dekorator obiektowy (klasyczny GoF)
3. 🐍 Dekorator Pythona (`@`, `functools.wraps`)
4. ⚙️ Dekoratory parametryzowane (fabryka dekoratorow)
5. 📦 Dekoratory z biblioteki standardowej (`lru_cache`, `singledispatch`)

## 1. 🔹 Problem: eksplozja klas bez dekoratora

Wyobrazmy sobie kawiarnie z napojami. Mamy Espresso, Latte, Cappuccino.
Chcemy dodac dodatki: mleko, karmel, bita smietana.

Bez dekoratora - dziedziczenie prowadzi do eksplozji klas:
- EspressoWithMilk, EspressoWithCaramel, EspressoWithMilkAndCaramel...
- 3 napoje x 4 kombinacje dodatkow = 12 klas (i rosnie wykladniczo!)

Dekorator (Decorator) - wzorzec strukturalny:
- Dodaje nowe zachowanie obiektowi BEZ zmiany jego klasy
- Owieja obiekt w inny obiekt tego samego interfejsu
- Mozna skladac wiele dekoratorow jak warstwy

> 💡 Reguła otwarte/zamkniete (OCP): klasy powinny byc otwarte
> na rozszerzanie, ale zamkniete na modyfikacje. Dekorator
> pozwala rozszerzac bez dziedziczenia.

In [ ]:
# Problem: eksplozja klas przez dziedziczenie
class Beverage:
    def cost(self) -> float: return 0.0
    def description(self) -> str: return 'Unknown'

class Espresso(Beverage):
    def cost(self) -> float: return 5.0
    def description(self) -> str: return 'Espresso'

# Zle podejscie: kazda kombinacja to osobna klasa
class EspressoWithMilk(Beverage):
    def cost(self) -> float: return 5.0 + 1.5
    def description(self) -> str: return 'Espresso + milk'

class EspressoWithMilkAndCaramel(Beverage):
    def cost(self) -> float: return 5.0 + 1.5 + 2.0
    def description(self) -> str: return 'Espresso + milk + caramel'

# Ile klas potrzebujemy? Wykladniczo wiele!
beverages = ['Espresso', 'Latte', 'Cappuccino']
extras = ['Milk', 'Caramel', 'Cream', 'Vanilla']
max_combinations = 2 ** len(extras)  # kazdy dodatek: jest lub nie ma
print(f'Napoje: {len(beverages)}')
print(f'Dodatki: {len(extras)}')
print(f'Max kombinacji na napoj: {max_combinations}')
print(f'Lacznie klas: {len(beverages) * max_combinations}')

---

### 🐍 Cwiczenia - problem eksplozji klas

1. Oblicz ile klas potrzeba dla systemu pizzy z 4 bazami i 8
   skladnikami przy dziedziczeniu.
2. Napisz klase `TextFormatter` z metodami `upper()`, `bold()`,
   `italic()`. Sprobuj rozszerzyc przez dziedziczenie o kazda kombinacje.
3. *(Trudniejsze)* Napisz funkcje `count_subclasses(n_bases, n_extras)`
   zwracajaca minimalna liczbe klas przy dziedziczeniu vs dekoratorach.

In [ ]:
# Cwiczenie 1: pizza
bases = ['Margherita', 'Pepperoni', 'Veggie', 'BBQ']
toppings = ['Cheese', 'Olives', 'Peppers', 'Onion', 'Mushrooms', 'Ham', 'Anchovy', 'Corn']
max_combos = ...
total = len(bases) * max_combos
print(f'Bazy: {len(bases)}, Skladniki: {len(toppings)}')
print(f'Klas bez dekoratora: {total}')

In [ ]:
# Cwiczenie 2: TextFormatter kombinacje
class TextFormatter:
    def __init__(self, text: str): self.text = text
    def upper(self) -> str: return self.text.upper()
    def bold(self) -> str: return f'**{self.text}**'
    def italic(self) -> str: return f'_{self.text}_'

# Ile klas trzeba by dodac dla kombinacji upper+bold, upper+italic itp.?
combinations = ['upper+bold', 'upper+italic', 'bold+italic', 'upper+bold+italic']
print(f'Klas do napisania: {len(combinations) + 3} (base + wszystkie kombinacje)')
print('Z dekoratorem: 3 dekoratory - skladamy je dynamicznie')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: kalkulator klas
def count_subclasses(n_bases: int, n_extras: int) -> dict:
    # hint: dziedziczenie = n_bases * 2**n_extras,
    # dekoratory = n_bases + n_extras
    ...

result = count_subclasses(4, 8)
print(result)

## 2. 🔹 Dekorator obiektowy (klasyczny GoF)

Klasyczny wzorzec Decorator z ksiazki Gang of Four:

Uczestnicy:
- Component (interfejs): definiuje wspolny interfejs
- ConcreteComponent: podstawowy obiekt
- Decorator: abstrakcyjny dekorator implementujacy Component,
  trzyma referencje do Component
- ConcreteDecorator: konkretny dekorator, dodaje zachowanie

Kluczowa wlasciwosc: Decorator implementuje ten sam interfejs
co dekorowany obiekt - klient nie wie, czy ma do czynienia
z oryginałem czy dekoratorem.

Mozna laczyc dekoratory w lancuch:
`CaramelDecorator(MilkDecorator(Espresso()))` - kolejnosc ma znaczenie!

In [ ]:
from abc import ABC, abstractmethod

class Beverage(ABC):
    @abstractmethod
    def cost(self) -> float: ...
    @abstractmethod
    def description(self) -> str: ...

class Espresso(Beverage):
    def cost(self) -> float: return 5.0
    def description(self) -> str: return 'Espresso'

class Latte(Beverage):
    def cost(self) -> float: return 7.0
    def description(self) -> str: return 'Latte'

class BeverageDecorator(Beverage):
    def __init__(self, beverage: Beverage):
        self._beverage = beverage
    def cost(self) -> float: return self._beverage.cost()
    def description(self) -> str: return self._beverage.description()

class MilkDecorator(BeverageDecorator):
    def cost(self) -> float: return self._beverage.cost() + 1.5
    def description(self) -> str: return self._beverage.description() + ' + milk'

class CaramelDecorator(BeverageDecorator):
    def cost(self) -> float: return self._beverage.cost() + 2.0
    def description(self) -> str: return self._beverage.description() + ' + caramel'

class CreamDecorator(BeverageDecorator):
    def cost(self) -> float: return self._beverage.cost() + 1.0
    def description(self) -> str: return self._beverage.description() + ' + cream'

# Skladamy dekoratory dynamicznie
drink1 = Espresso()
drink2 = MilkDecorator(Espresso())
drink3 = CaramelDecorator(MilkDecorator(Espresso()))
drink4 = CreamDecorator(CaramelDecorator(MilkDecorator(Latte())))

for d in [drink1, drink2, drink3, drink4]:
    print(f'{d.description()}: {d.cost()} PLN')

---

### 🐍 Cwiczenia - dekorator obiektowy

1. Dodaj dekorator `VanillaDecorator` (+1.0 PLN) do systemu napojow.
   Zbuduj drink: Latte + vanilla + milk.
2. Napisz hierarchie `Shape` z `Circle` i `Rectangle`. Dekoratory:
   `ColorDecorator(color)` i `BorderDecorator(width)`. Interfejs:
   `draw() -> str`.
3. *(Trudniejsze)* Napisz `PriceHistoryDecorator` ktory sledzi
   wszystkie dekoratory uzyte na napoju i zwraca ich liste.

In [ ]:
# Cwiczenie 1: VanillaDecorator
class VanillaDecorator(BeverageDecorator):
    def cost(self) -> float: ...
    def description(self) -> str: ...

drink = VanillaDecorator(MilkDecorator(Latte()))
print(drink.description(), '->', drink.cost())

In [ ]:
# Cwiczenie 2: Shape + dekoratory
class Shape(ABC):
    @abstractmethod
    def draw(self) -> str: ...

class Circle(Shape):
    def draw(self) -> str: return 'Circle'

class Rectangle(Shape):
    def draw(self) -> str: return 'Rectangle'

class ShapeDecorator(Shape):
    def __init__(self, shape: Shape): self._shape = shape
    def draw(self) -> str: return self._shape.draw()

class ColorDecorator(ShapeDecorator):
    def __init__(self, shape: Shape, color: str):
        super().__init__(shape)
        self._color = color
    def draw(self) -> str: ...

class BorderDecorator(ShapeDecorator):
    def __init__(self, shape: Shape, width: int):
        super().__init__(shape)
        self._width = width
    def draw(self) -> str: ...

shape = BorderDecorator(ColorDecorator(Circle(), 'red'), 2)
print(shape.draw())  # Circle[red][border=2]

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: PriceHistoryDecorator
class PriceHistoryDecorator(BeverageDecorator):
    # hint: zbieraj nazwy klas dekoratorow w liscie,
    # przekazuj liste w dol przez lancuch dekoratorow
    def get_layers(self) -> list:
        ...

pd = PriceHistoryDecorator(CaramelDecorator(MilkDecorator(Espresso())))
print(pd.description())
print('Warstwy:', pd.get_layers())

## 3. 🔹 Dekorator Pythona (`@`, `functools.wraps`)

Python ma wbudowana skladnie `@dekorator` - to cukier syntaktyczny
dla wywolania `func = dekorator(func)`.

Dekorator funkcji to funkcja ktora:
1. Przyjmuje funkcje jako argument
2. Zwraca nowa funkcje (zazwyczaj `wrapper`)
3. Wrapper wywoluje oryginal, dodajac zachowanie

`functools.wraps` - zachowuje metadane oryginalnej funkcji:
- `__name__` - nazwa funkcji
- `__doc__` - dokumentacja
- `__annotations__` - adnotacje typow

Bez `@functools.wraps` debugowanie jest utrudnione - wszystkie
zdekorowane funkcje wygladalyby jak `wrapper`.

In [ ]:
import functools
import time

# Dekorator bez functools.wraps - traci metadane
def bad_timer(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f'{func.__name__}: {time.perf_counter() - start:.4f}s')
        return result
    return wrapper

# Dekorator z functools.wraps - zachowuje metadane
def timer(func):
    @functools.wraps(func)  # kopiuje __name__, __doc__, __annotations__
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f'{func.__name__}: {time.perf_counter() - start:.4f}s')
        return result
    return wrapper

def logger(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f'Calling {func.__name__}({args}, {kwargs})')
        result = func(*args, **kwargs)
        print(f'{func.__name__} returned {result}')
        return result
    return wrapper

@timer
@logger
def compute(n: int) -> int:
    """Oblicza sume 0..n-1."""
    return sum(range(n))

compute(1_000_000)
print('Nazwa:', compute.__name__)   # compute (nie wrapper)
print('Docs:', compute.__doc__)     # zachowana dokumentacja

---

### 🐍 Cwiczenia - dekorator funkcji

1. Napisz dekorator `@log_calls` drukujacy argumenty przed wywolaniem
   i wynik po. Zastosuj do 3 roznych funkcji.
2. Napisz dekorator `@timer` mierzacy czas i drukujacy `func: X.XXXXs`.
   Zastosuj do funkcji z petla `sum(range(n))`.
3. *(Trudniejsze)* Napisz dekorator `@counted` ktory zlicza wywolania
   funkcji i umieszcza licznik w atrybucie `func.call_count`.

In [ ]:
# Cwiczenie 1: @log_calls
def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        ...
    return wrapper

@log_calls
def add(a: int, b: int) -> int:
    return a + b

@log_calls
def greet(name: str) -> str:
    return f'Hello, {name}!'

add(3, 4)
greet('Alice')
print(add.__name__)  # add

In [ ]:
# Cwiczenie 2: @timer
def timer_dec(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        ...
    return wrapper

@timer_dec
def slow_sum(n: int) -> int:
    return sum(range(n))

result = slow_sum(10_000_000)
print(f'Result: {result}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: @counted
def counted(func):
    # hint: uzyj atrybutu wrapper.call_count = 0 po definicji wrappera
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        ...
    wrapper.call_count = 0
    return wrapper

@counted
def multiply(a, b): return a * b

for i in range(5):
    multiply(i, 2)
print(f'Wywolano: {multiply.call_count} razy')

## 4. 🔹 Dekoratory parametryzowane (fabryka dekoratorow)

Dekorator parametryzowany to funkcja zwracajaca dekorator.
Struktura trzech poziomow zagniezdzen:

```
def decorator_factory(param):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # uzywa param
            return func(*args, **kwargs)
        return wrapper
    return decorator
```

Skladnia: `@decorator_factory(param)` - najpierw wywoluje fabryke
z parametrami, potem uzywa zwroconego dekoratora.

Przyklady z praktyki:
- `@retry(max_attempts=3, delay=0.1)` - ponowne proby
- `@validate(min_val=0, max_val=100)` - walidacja argumentow
- `@cache(ttl=60)` - cache z czasem zycia
- `@rate_limit(calls=10, period=60)` - limitowanie czestotliwosci

In [ ]:
import functools
import time

def retry(max_attempts: int = 3, delay: float = 0.1):
    """Fabryka dekoratora ponownych prob."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exc = e
                    print(f'Attempt {attempt}/{max_attempts} failed: {e}')
                    if attempt < max_attempts:
                        time.sleep(delay)
            raise last_exc
        return wrapper
    return decorator

def validate(min_val, max_val):
    """Fabryka dekoratora walidacji pierwszego argumentu."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(value, *args, **kwargs):
            if not (min_val <= value <= max_val):
                raise ValueError(f'{value} not in [{min_val}, {max_val}]')
            return func(value, *args, **kwargs)
        return wrapper
    return decorator

import random

@retry(max_attempts=3, delay=0.01)
def unstable_api(n: int) -> str:
    if random.random() < 0.6:
        raise ConnectionError('timeout')
    return f'ok({n})'

@validate(0, 100)
def set_volume(level: int) -> str:
    return f'Volume: {level}%'

try:
    print(unstable_api(1))
except ConnectionError:
    print('All attempts failed')

print(set_volume(50))
try:
    set_volume(150)
except ValueError as e:
    print(f'Blad: {e}')

---

### 🐍 Cwiczenia - dekoratory parametryzowane

1. Napisz `@validate(min_val, max_val)` walidujacy pierwszy argument
   numeryczny. Rzuc `ValueError` jesli poza zakresem.
2. Napisz `@prefix(text)` dodajacy prefix do wyniku funkcji
   zwracajacej string.
3. *(Trudniejsze)* Napisz `@rate_limit(max_calls, period_seconds)`
   rzucajacy `RuntimeError` gdy przekroczono limit wywolan.

In [ ]:
# Cwiczenie 1: @validate
def validate_range(min_val, max_val):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(value, *args, **kwargs):
            ...
        return wrapper
    return decorator

@validate_range(1, 31)
def set_day(day: int) -> str:
    return f'Day: {day}'

print(set_day(15))
try:
    set_day(32)
except ValueError as e:
    print(e)

In [ ]:
# Cwiczenie 2: @prefix
def prefix(text: str):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            ...
        return wrapper
    return decorator

@prefix('[INFO] ')
def get_status() -> str:
    return 'System OK'

@prefix('[ERROR] ')
def get_error() -> str:
    return 'Connection failed'

print(get_status())
print(get_error())

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: @rate_limit
def rate_limit(max_calls: int, period_seconds: float):
    # hint: trzymaj timestamps wywolan w liscie,
    # usuwaj te starsze niz period_seconds
    def decorator(func):
        calls = []
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            now = time.time()
            ...
        return wrapper
    return decorator

@rate_limit(3, 1.0)  # max 3 wywolania na sekunde
def api_call(n: int) -> str:
    return f'response_{n}'

for i in range(5):
    try:
        print(api_call(i))
    except RuntimeError as e:
        print(f'Rate limit: {e}')

## 5. 🔹 Dekoratory z biblioteki standardowej

`functools.lru_cache` - memoizacja z LRU (Least Recently Used):
- `@lru_cache(maxsize=128)` - cache do 128 wynikow
- `@lru_cache(maxsize=None)` - nieograniczony cache (alias: `@cache`)
- Metoda `cache_info()` - statystyki (hits, misses, currsize)
- Metoda `cache_clear()` - czyszczenie cache
- Dziala tylko z argumentami hashowalnymi

`functools.singledispatch` - polimorfizm oparty na typie argumentu:
- Zamienia zwykla funkcje w funkcje generyczna
- Rejestruje implementacje dla roznych typow przez `@func.register`
- Wywoluje implementacje odpowiednia dla typu pierwszego argumentu
- Dla nieznanego typu uzywa domyslnej implementacji

Inne wazne dekoratory:
- `@property` - getter/setter
- `@classmethod`, `@staticmethod` - metody klasowe
- `@dataclasses.dataclass` - generacja `__init__`, `__repr__`

In [ ]:
import functools

# functools.lru_cache - memoizacja
@functools.lru_cache(maxsize=128)
def fibonacci(n: int) -> int:
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(fibonacci(40))
print('Cache info:', fibonacci.cache_info())  # hits, misses, maxsize, currsize

# Porownanie z cache i bez
import time

def fib_slow(n: int) -> int:
    if n < 2: return n
    return fib_slow(n-1) + fib_slow(n-2)

start = time.perf_counter()
fib_slow(30)
print(f'Bez cache: {time.perf_counter()-start:.3f}s')

fibonacci.cache_clear()
start = time.perf_counter()
fibonacci(30)
print(f'Z cache:   {time.perf_counter()-start:.6f}s')


# functools.singledispatch - polimorfizm wg typu
@functools.singledispatch
def format_value(value) -> str:
    return str(value)  # domyslna implementacja

@format_value.register(int)
def _(value: int) -> str:
    return f'{value:,}'

@format_value.register(float)
def _(value: float) -> str:
    return f'{value:.2f}'

@format_value.register(list)
def _(value: list) -> str:
    return '[' + ', '.join(format_value(v) for v in value) + ']'

print(format_value(1234567))          # 1,234,567
print(format_value(3.14159))          # 3.14
print(format_value([1, 2.5, 'abc']))  # [1, 2.50, abc]
print(format_value('hello'))          # hello (domyslna)

---

### 🐍 Cwiczenia - dekoratory biblioteczne

1. Uzyj `@lru_cache` do optymalizacji funkcji `coin_change(coins, amount)`
   (minimalna liczba monet). Porownaj czas z cache i bez.
2. Napisz `@singledispatch` dla `serialize(value)` obsługujacej
   `int`, `float`, `str`, `list`, `dict`.
3. *(Trudniejsze)* Zaimplementuj wlasny `@memoize(max_size)` z kolejka
   FIFO (gdy przekroczy max_size usuwa najstarszy wpis) uzywajac
   `collections.OrderedDict`.

In [ ]:
# Cwiczenie 1: lru_cache dla coin_change
@functools.lru_cache(maxsize=None)
def coin_change(coins: tuple, amount: int) -> int:
    if amount == 0: return 0
    if amount < 0: return float('inf')
    return ...

coins = (1, 5, 10, 25)
print(coin_change(coins, 41))  # 4 monety: 25+10+5+1
print('Cache:', coin_change.cache_info())

In [ ]:
# Cwiczenie 2: singledispatch serialize
@functools.singledispatch
def serialize(value) -> str:
    raise TypeError(f'Unsupported type: {type(value)}')

@serialize.register(int)
def _(v): ...

@serialize.register(float)
def _(v): ...

@serialize.register(str)
def _(v): ...

@serialize.register(list)
def _(v): ...

@serialize.register(dict)
def _(v): ...

print(serialize(42))
print(serialize(3.14))
print(serialize('hello'))
print(serialize([1, 2, 3]))
print(serialize({'a': 1}))

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: @memoize z FIFO
import collections

def memoize(max_size: int = 128):
    # hint: uzyj collections.OrderedDict, przy przekroczeniu
    # max_size wywolaj cache.popitem(last=False)
    def decorator(func):
        cache = collections.OrderedDict()
        @functools.wraps(func)
        def wrapper(*args):
            ...
        wrapper.cache = cache
        return wrapper
    return decorator

@memoize(max_size=50)
def fib(n: int) -> int:
    if n < 2: return n
    return fib(n-1) + fib(n-2)

start = time.perf_counter()
print(f'fib(35) = {fib(35)}')
print(f'Time: {time.perf_counter()-start:.6f}s')
print(f'Cache size: {len(fib.cache)}')